# Circle Diffusion — Post-Processing & Evaluation

This notebook loads a trained checkpoint and TensorBoard logs from a circle
diffusion experiment, visualises training dynamics, generates samples, and
computes quantitative quality metrics.

**Experiment directory**: `outputs/EXP-001_circle_radial_baseline/`

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml

# Ensure src/ is importable
sys.path.insert(0, str(Path(".").resolve() / "src"))
sys.path.insert(0, str(Path(".").resolve() / "scripts"))

from graph_diffusion.postprocessing import load_checkpoint, read_tensorboard_logs
from metrics import (
    compute_radii_stats,
    compute_smoothness,
    compute_closure_error,
    compute_boundary_violations,
    compute_circularity,
    ks_statistic,
    extract_sorted_radii,
)

%matplotlib inline

SyntaxError: invalid syntax (1679942856.py, line 14)

## 1. Configuration

In [ ]:
EXPERIMENT_DIR = Path("outputs/EXP-001_circle_radial_baseline")
CONFIG_PATH = "configs/circle_radial.yaml"
N_SAMPLES = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

clamp_cfg = config.get("clamp_range")
clamp_range = tuple(clamp_cfg) if clamp_cfg else None
r_min = clamp_cfg[0] if clamp_cfg else 0.5
r_max = clamp_cfg[1] if clamp_cfg else 1.5

print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"Device: {DEVICE}")
print(f"Clamp range: [{r_min}, {r_max}]")

## 2. Load Checkpoint

In [ ]:
checkpoint = load_checkpoint(str(EXPERIMENT_DIR / "checkpoint.pt"))

## 3. Training Loss Curves

### 3a. From TensorBoard Logs

In [ ]:
tb_dir = EXPERIMENT_DIR / "tensorboard"

if tb_dir.exists():
    tensorboard_data = read_tensorboard_logs(str(tb_dir))
else:
    print(f"No TensorBoard logs found at {tb_dir}")
    tensorboard_data = {}

In [ ]:
def plot_losses(tensorboard_data):
    """Plot training and validation losses from TensorBoard data."""
    if "Loss/train" in tensorboard_data and "Loss/test" in tensorboard_data:
        train_steps, train_losses = zip(*tensorboard_data["Loss/train"])
        test_steps, test_losses = zip(*tensorboard_data["Loss/test"])

        plt.figure(figsize=(10, 5))
        plt.plot(train_steps, train_losses, label="Training Loss", color="blue")
        plt.plot(test_steps, test_losses, label="Validation Loss", color="orange")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training and Validation Losses")
        plt.legend()
        plt.grid()
        plt.show()
    elif "Loss/train" in tensorboard_data:
        train_steps, train_losses = zip(*tensorboard_data["Loss/train"])

        plt.figure(figsize=(10, 5))
        plt.plot(train_steps, train_losses, label="Training Loss", color="blue")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training Loss")
        plt.legend()
        plt.grid()
        plt.show()
    else:
        print("No training or validation loss data available for plotting.")


plot_losses(tensorboard_data)

In [ ]:
def plot_log_losses(tensorboard_data):
    """Plot training and validation losses on log scale from TensorBoard data."""
    if "Loss/train" in tensorboard_data and "Loss/test" in tensorboard_data:
        train_steps, train_losses = zip(*tensorboard_data["Loss/train"])
        test_steps, test_losses = zip(*tensorboard_data["Loss/test"])

        plt.figure(figsize=(10, 5))
        plt.plot(train_steps, np.log(train_losses), label="Training Loss", color="blue")
        plt.plot(test_steps, np.log(test_losses), label="Validation Loss", color="orange")
        plt.xlabel("Epoch")
        plt.ylabel("log(Loss)")
        plt.title("Training and Validation Losses (Log Scale)")
        plt.legend()
        plt.grid()
        plt.show()
    elif "Loss/train" in tensorboard_data:
        train_steps, train_losses = zip(*tensorboard_data["Loss/train"])

        plt.figure(figsize=(10, 5))
        plt.plot(train_steps, np.log(train_losses), label="Training Loss", color="blue")
        plt.xlabel("Epoch")
        plt.ylabel("log(Loss)")
        plt.title("Training Loss (Log Scale)")
        plt.legend()
        plt.grid()
        plt.show()
    else:
        print("No training or validation loss data available for plotting.")


plot_log_losses(tensorboard_data)

### 3b. From loss_log.json (fallback)

In [ ]:
import json

loss_log_path = EXPERIMENT_DIR / "loss_log.json"
if loss_log_path.exists():
    with open(loss_log_path) as f:
        loss_log = json.load(f)

    epochs = [e["epoch"] for e in loss_log]
    train_loss = [e["train_loss"] for e in loss_log]
    val_loss = [e["val_loss"] for e in loss_log]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, train_loss, label="Train", color="blue")
    ax1.plot(epochs, val_loss, label="Validation", color="orange")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss (linear)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, train_loss, label="Train", color="blue")
    ax2.plot(epochs, val_loss, label="Validation", color="orange")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.set_title("Loss (log scale)")
    ax2.set_yscale("log")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No loss_log.json found.")

## 4. Reconstruct Model & Generate Samples

In [ ]:
from graph_diffusion.building_blocks.noise_schedule import NoiseSchedule
from graph_diffusion.data.circledataset import UnitCircleDataset
from graph_diffusion.data.transforms import ComputeAngularEdgeFeatures
from graph_diffusion.model.graph_diffusion_model import GraphDiffusionModel
from graph_diffusion.model.score_network import ScoreNetwork

# Build model from config
ns_cfg = config.get("noise_schedule", {})
noise_schedule = NoiseSchedule(
    T=ns_cfg.get("T", 200),
    schedule_type=ns_cfg.get("schedule_type", "cosine"),
    beta_start=ns_cfg.get("beta_start", 1e-4),
    beta_end=ns_cfg.get("beta_end", 0.02),
)

sn_cfg = config.get("score_network", {})
mlp_cfg = config.get("mlp", {})
score_network = ScoreNetwork(
    node_dim=sn_cfg.get("node_dim", 32),
    edge_dim=sn_cfg.get("edge_dim", 2),
    global_dim=sn_cfg.get("global_dim", 8),
    time_embed_dim=sn_cfg.get("time_embed_dim", 64),
    n_layers=sn_cfg.get("n_layers", 4),
    hidden_dims=sn_cfg.get("hidden_dims", [64, 64]),
    activation=mlp_cfg.get("activation", "silu"),
    layer_norm=mlp_cfg.get("layer_norm", True),
    residual=mlp_cfg.get("residual", True),
    input_dim=sn_cfg.get("input_dim", 1),
)

model = GraphDiffusionModel(
    score_network=score_network,
    noise_schedule=noise_schedule,
)

device = torch.device(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params:,} parameters on {device}")

In [ ]:
# Load dataset for reference statistics and template graph
ds_cfg = config.get("circle_dataset", {})
pre_transform = ComputeAngularEdgeFeatures()

dataset = UnitCircleDataset(
    root="data/circle",
    n_graphs=ds_cfg.get("n_graphs", 2000),
    n_nodes=ds_cfg.get("n_nodes", 64),
    n_fourier_modes=ds_cfg.get("n_fourier_modes", 5),
    amplitude_scale=ds_cfg.get("amplitude_scale", 0.15),
    r_min=ds_cfg.get("r_min", 0.5),
    r_max=ds_cfg.get("r_max", 1.5),
    k_neighbors=ds_cfg.get("k_neighbors", 2),
    global_dim=ds_cfg.get("global_dim", 8),
    seed=ds_cfg.get("seed", 42),
    pre_transform=pre_transform,
)

template = dataset[0].to(device)
print(f"Dataset: {len(dataset)} graphs, {ds_cfg.get('n_nodes', 64)} nodes each")

In [ ]:
# Generate samples
all_radii = []
per_sample_metrics = []

print(f"Generating {N_SAMPLES} samples...")
with torch.no_grad():
    for i in range(N_SAMPLES):
        torch.manual_seed(i)
        result = model.sample(template, clamp_range=clamp_range)
        r_sorted, theta_sorted = extract_sorted_radii(result, template)
        all_radii.append(r_sorted)

        metrics = {
            "sample": i,
            "radii_stats": compute_radii_stats(r_sorted),
            "smoothness": compute_smoothness(r_sorted),
            "closure_error": compute_closure_error(r_sorted),
            "boundary_violations": compute_boundary_violations(
                r_sorted, r_min, r_max
            ),
            "circularity_cv": compute_circularity(r_sorted),
        }
        per_sample_metrics.append(metrics)

print(f"Generated {len(all_radii)} samples.")

## 5. Reference vs Generated — Radii Distribution

In [ ]:
# Collect reference radii from training data
ref_radii = []
for i in range(min(200, len(dataset))):
    ref_radii.append(dataset[i].x[:, 0].numpy())
ref_radii_all = np.concatenate(ref_radii)
gen_radii_all = np.concatenate(all_radii)

ref_stats = compute_radii_stats(ref_radii_all)
gen_stats = compute_radii_stats(gen_radii_all)

print("Reference (training data) radii stats:")
for k, v in ref_stats.items():
    print(f"  {k}: {v:.4f}")

print("\nGenerated samples radii stats:")
for k, v in gen_stats.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Radii histogram overlay
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ref_radii_all, bins=60, alpha=0.5, density=True, label="Training data", color="blue")
ax.hist(gen_radii_all, bins=60, alpha=0.5, density=True, label="Generated", color="orange")
ax.axvline(1.0, color="k", linestyle="--", alpha=0.5, label="Unit circle")
ax.set_xlabel("Radius")
ax.set_ylabel("Density")
ax.set_title("Radii Distribution: Training vs Generated")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. Quality Metrics Summary

In [ ]:
avg_smoothness = np.mean([m["smoothness"] for m in per_sample_metrics])
avg_closure = np.mean([m["closure_error"] for m in per_sample_metrics])
avg_cv = np.mean([m["circularity_cv"] for m in per_sample_metrics])
avg_violations = np.mean(
    [m["boundary_violations"]["total"] for m in per_sample_metrics]
)
ks = ks_statistic(ref_radii_all, gen_radii_all)

print(f"Quality metrics (averaged over {N_SAMPLES} samples):")
print(f"  Smoothness (2nd-order diff):  {avg_smoothness:.4f}")
print(f"  Closure error (|r_0 - r_N|): {avg_closure:.4f}")
print(f"  Circularity (CV of radii):   {avg_cv:.4f}")
print(f"  Boundary violation rate:     {avg_violations:.4f}")
print(f"  KS statistic (ref vs gen):   {ks:.4f}")

## 7. Sample Gallery

In [ ]:
theta = np.arctan2(
    template.pos[:, 1].cpu().numpy(),
    template.pos[:, 0].cpu().numpy(),
)
order = np.argsort(theta)
theta_sorted = theta[order]

n_show = min(16, len(all_radii))
cols = 4
rows = (n_show + cols - 1) // cols
t_ref = np.linspace(0, 2 * np.pi, 100)

fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
axes = np.array(axes).flatten()

for i in range(n_show):
    ax = axes[i]
    r = all_radii[i]
    x_cart = r * np.cos(theta_sorted)
    y_cart = r * np.sin(theta_sorted)
    x_cart = np.append(x_cart, x_cart[0])
    y_cart = np.append(y_cart, y_cart[0])

    ax.plot(np.cos(t_ref), np.sin(t_ref), "k--", alpha=0.3, linewidth=0.5)
    ax.plot(x_cart, y_cart, "b-", linewidth=1.2)
    ax.set_aspect("equal")
    ax.set_title(f"Sample {i + 1}", fontsize=9)
    ax.grid(True, alpha=0.2)
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)

for i in range(n_show, len(axes)):
    axes[i].set_visible(False)

fig.suptitle("Generated Shape Gallery", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Smoothness & Circularity Distributions

In [ ]:
smoothness_vals = [m["smoothness"] for m in per_sample_metrics]
cv_vals = [m["circularity_cv"] for m in per_sample_metrics]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.hist(smoothness_vals, bins=25, edgecolor="black", alpha=0.7)
ax1.set_xlabel("Smoothness (2nd-order diff)")
ax1.set_ylabel("Count")
ax1.set_title("Smoothness Distribution")
ax1.grid(True, alpha=0.3)

ax2.hist(cv_vals, bins=25, edgecolor="black", alpha=0.7, color="orange")
ax2.set_xlabel("Circularity (CV of radii)")
ax2.set_ylabel("Count")
ax2.set_title("Circularity Distribution")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Save Evaluation Report

In [ ]:
import json

report = {
    "reference_stats": ref_stats,
    "generated_stats": gen_stats,
    "aggregate_metrics": {
        "smoothness": float(avg_smoothness),
        "closure_error": float(avg_closure),
        "circularity_cv": float(avg_cv),
        "boundary_violation_rate": float(avg_violations),
        "ks_statistic": float(ks),
    },
    "n_samples": N_SAMPLES,
    "epochs": checkpoint.get("epoch", checkpoint.get("epochs", "?")),
}

report_path = EXPERIMENT_DIR / "evaluation_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Saved evaluation report to {report_path}")